# Baseline + Recall Measurement

Establishing where a standard embedding model lands on CoIR APPS, and whether a two-stage architecture with execution reranking is worth building.

The question this notebook answers: **is the correct snippet inside the top-K often enough for a second pass to matter?**

### 0. Check the runtime

GPU is for measurement speed only. The shipped solution must run on CPU.

In [1]:
!nvidia-smi

Sat Sep 19 23:15:54 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

### 1. Install dependencies

- `sentence-transformers` for loading and running embedding models
- `datasets` for pulling CoIR APPS from HuggingFace

In [2]:
!pip install -q sentence-transformers datasets

### 2. Load the dataset

Three pieces:

- **corpus** — the Python solutions being searched
- **queries** — contest problem statements in English
- **qrels** — which query maps to which solution

Filtering to the test partition, since that's what gets scored. All three should come back at 3,765.

The qrels are strictly one-to-one and numerically aligned (`q5001 → d5001`), so every query has exactly one correct answer and no false negatives exist.

In [3]:
from datasets import load_dataset
import pandas as pd

corpus  = load_dataset("CoIR-Retrieval/apps", "corpus",  split="corpus").to_pandas()
queries = load_dataset("CoIR-Retrieval/apps", "queries", split="queries").to_pandas()
qrels   = load_dataset("CoIR-Retrieval/apps", "default", split="test").to_pandas()

corpus_t  = corpus[corpus.partition  == "test"].reset_index(drop=True)
queries_t = queries[queries.partition == "test"].reset_index(drop=True)

print("corpus test :", len(corpus_t))
print("queries test:", len(queries_t))
print("qrels test  :", len(qrels))
print()
print(qrels.head(3))

README.md:   0%|          | 0.00/2.60k [00:00<?, ?B/s]

corpus/corpus-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 2.70MB            

corpus/corpus-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating corpus split:   0%|          | 0/8765 [00:00<?, ? examples/s]

queries/queries-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 6.61MB            

...

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 43.7kB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/5000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3765 [00:00<?, ? examples/s]

corpus test : 3765
queries test: 3765
qrels test  : 3765

  query-id corpus-id  score
0    q5001     d5001      1
1    q5002     d5002      1
2    q5003     d5003      1


### 3. Load the embedding model

`e5-base-v2`, 110M parameters. Chosen because the CoIR paper puts it at 11.52 NDCG@10 on APPS, the best of the small models.

Code-specific models were ruled out on evidence: UniXcoder scores 1.36 here, worse than generic text models. Code embedders train on docstring-to-function pairs, which is a different distribution from contest problems.

512 token cap, which truncates 27% of queries.

In [4]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("intfloat/e5-base-v2", device="cuda")
model.max_seq_length = 512

print("max_seq_length:", model.max_seq_length)
print("embedding dim :", model.get_sentence_embedding_dimension())

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/67.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/650 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  438MB            

...

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

max_seq_length: 512
embedding dim : 768


/tmp/ipykernel_3298/58323906.py:7: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("embedding dim :", model.get_sentence_embedding_dimension())


### 4. Encode queries and corpus

E5 requires prefixes. `query:` on queries, `passage:` on documents. Omitting them costs several points.

Embeddings are normalized so cosine similarity reduces to a dot product.

Both timings matter. Corpus encoding is the index-build cost, which is what P1 (retrieval across versions) is judged on.

In [5]:
import time

q_texts = ["query: "   + t for t in queries_t.text]
c_texts = ["passage: " + t for t in corpus_t.text]

t0 = time.time()
q_emb = model.encode(q_texts, batch_size=64, normalize_embeddings=True,
                     show_progress_bar=True, convert_to_numpy=True)
t_q = time.time() - t0

t0 = time.time()
c_emb = model.encode(c_texts, batch_size=64, normalize_embeddings=True,
                     show_progress_bar=True, convert_to_numpy=True)
t_c = time.time() - t0

print(f"\nqueries encoded in {t_q:.1f}s")
print(f"corpus  encoded in {t_c:.1f}s")
print("shapes:", q_emb.shape, c_emb.shape)

Batches:   0%|          | 0/59 [00:00<?, ?it/s]

Batches:   0%|          | 0/59 [00:00<?, ?it/s]


queries encoded in 105.6s
corpus  encoded in 67.5s
shapes: (3765, 768) (3765, 768)


### 5. Score and evaluate

Full similarity matrix, 3,765 × 3,765, then the rank of the gold document per query.

Since there's exactly one binary-relevant document per query, ideal DCG is always 1.0, so NDCG@10 collapses to `1 / log2(rank + 1)` when the gold lands in the top 10, and 0 otherwise. NDCG and MRR move together here.

Recall@K is the number that matters: it sets the ceiling for anything a second stage can achieve.

In [6]:
import numpy as np

# gold mapping
gold = dict(zip(qrels["query-id"], qrels["corpus-id"]))
cid_to_idx = {cid: i for i, cid in enumerate(corpus_t._id)}
gold_idx = np.array([cid_to_idx[gold[qid]] for qid in queries_t._id])

# similarity (normalized embeddings -> dot product is cosine)
sim = q_emb @ c_emb.T          # 3765 x 3765

# rank of the gold doc for each query
gold_score = sim[np.arange(len(sim)), gold_idx]
rank = (sim > gold_score[:, None]).sum(axis=1) + 1   # 1-indexed

print("=== RECALL ===")
for k in [1, 5, 10, 50, 100, 200, 500]:
    print(f"  Recall@{k:<4} {(rank <= k).mean()*100:6.2f}%")

ndcg10 = np.where(rank <= 10, 1/np.log2(rank + 1), 0).mean()
mrr    = (1 / rank).mean()

print("\n=== METRICS ===")
print(f"  NDCG@10  {ndcg10*100:.2f}")
print(f"  MRR      {mrr*100:.2f}")
print(f"\n  median rank of gold: {int(np.median(rank))}")

=== RECALL ===
  Recall@1      8.50%
  Recall@5     14.85%
  Recall@10    19.04%
  Recall@50    31.29%
  Recall@100   38.57%
  Recall@200   48.07%
  Recall@500   62.82%

=== METRICS ===
  NDCG@10  13.11
  MRR      12.10

  median rank of gold: 228


### 6. Ceiling for execution reranking

Competitive programming problems ship with example input and expected output. 78% of queries have a parseable example block.

That opens a different kind of reranking: instead of measuring similarity, run the candidate and check whether it produces the expected output. A snippet that outputs correctly is almost certainly the answer.

This cell computes the best case, assuming execution works perfectly and promotes the gold to rank 1 whenever it's inside the top-K.

In [7]:
import re
pat = re.compile(r'-----Examples?-----\s*Input\s*\n(.*?)\n\s*Output\s*\n(.*?)(?:\n\s*Input\s*\n|\n-----|\Z)', re.S)
has_ex = np.array([bool(pat.search(t)) for t in queries_t.text])
print(f"queries with parseable example: {has_ex.mean()*100:.1f}%\n")

for K in [50, 100, 200, 500]:
    in_top = rank <= K
    # assume: gold in top-K AND has example -> execution promotes it to rank 1
    fixed = in_top & has_ex
    new_rank = np.where(fixed, 1, rank)
    n = np.where(new_rank <= 10, 1/np.log2(new_rank+1), 0).mean()
    print(f"K={K:<4} ceiling NDCG@10 = {n*100:5.2f}   (fires on {fixed.mean()*100:.1f}% of queries)")

queries with parseable example: 78.0%

K=50   ceiling NDCG@10 = 26.28   (fires on 21.3% of queries)
K=100  ceiling NDCG@10 = 31.86   (fires on 26.9% of queries)
K=200  ceiling NDCG@10 = 39.43   (fires on 34.4% of queries)
K=500  ceiling NDCG@10 = 51.33   (fires on 46.3% of queries)


### 7. Does the gold snippet actually run?

The ceiling above assumes gold snippets execute correctly on their own example input. Untested.

These are scraped accepted solutions reading from stdin. If the harness feeds input differently than expected, or output formatting differs, everything fails and the idea is worthless.

Testing 30 pairs before building anything on top.

In [8]:
import subprocess, tempfile, os, re

def run_snippet(code, stdin_data, timeout=2):
    with tempfile.NamedTemporaryFile('w', suffix='.py', delete=False) as f:
        f.write(code); path = f.name
    try:
        r = subprocess.run(['python3', path], input=stdin_data,
                           capture_output=True, text=True, timeout=timeout)
        return r.stdout.strip()
    except Exception:
        return None
    finally:
        os.unlink(path)

# --- test A: does the GOLD snippet pass its own example? ---
idx = np.where(has_ex)[0][:30]
passes = 0
for i in idx:
    m = pat.search(queries_t.text[i])
    inp, exp = m.group(1).strip(), m.group(2).strip()
    out = run_snippet(corpus_t.text[gold_idx[i]], inp + "\n")
    if out is not None and out == exp:
        passes += 1
print(f"A) gold passes own example: {passes}/30")

A) gold passes own example: 22/30


### 8. How many wrong programs pass?

The real threat to execution reranking.

Many of these problems output `YES`/`NO` or a small integer. If dozens of wrong snippets pass a single test case by chance, the signal is too noisy to rank on.

Running 100 random wrong snippets per query against the example, counting how many slip through.

- under 2 → near-unique identifier, build it
- 2 to 10 → strong signal, combine with embedding score
- over 20 → too noisy at high K, would need multiple test cases

In [9]:
import random
random.seed(0)

test_idx = np.where(has_ex)[0][:15]
results = []

for i in test_idx:
    m = pat.search(queries_t.text[i])
    inp, exp = m.group(1).strip(), m.group(2).strip()

    # does gold pass?
    gold_ok = run_snippet(corpus_t.text[gold_idx[i]], inp + "\n") == exp
    if not gold_ok:
        continue

    # how many of 100 random wrong snippets also pass?
    cands = random.sample([j for j in range(len(corpus_t)) if j != gold_idx[i]], 100)
    fp = sum(1 for j in cands
             if run_snippet(corpus_t.text[j], inp + "\n") == exp)
    results.append(fp)
    print(f"query {i}: {fp}/100 wrong snippets also passed")

print(f"\nmean false positives per 100 candidates: {np.mean(results):.1f}")

query 0: 0/100 wrong snippets also passed
query 1: 0/100 wrong snippets also passed
query 2: 1/100 wrong snippets also passed
query 3: 1/100 wrong snippets also passed
query 5: 0/100 wrong snippets also passed
query 6: 0/100 wrong snippets also passed
query 7: 0/100 wrong snippets also passed
query 8: 0/100 wrong snippets also passed
query 9: 1/100 wrong snippets also passed
query 10: 0/100 wrong snippets also passed
query 12: 1/100 wrong snippets also passed
query 14: 0/100 wrong snippets also passed

mean false positives per 100 candidates: 0.3


### 9. Timing the execution rerank

How expensive is Stage 2 in practice?

- Takes the top-K candidates from the embedding search
- Runs each against the example input, in parallel threads
- Records wall-clock time, how many passed, and whether gold was found

Speed is explicitly judged, so this decides how wide K can go.

In [10]:
import time
from concurrent.futures import ThreadPoolExecutor

def rerank_timed(qi, K=200, workers=8, timeout=2):
    m = pat.search(queries_t.text[qi])
    if not m: return None
    inp, exp = m.group(1).strip(), m.group(2).strip()

    cand = np.argsort(-sim[qi])[:K]
    t0 = time.time()
    with ThreadPoolExecutor(max_workers=workers) as ex:
        outs = list(ex.map(lambda j: run_snippet(corpus_t.text[j], inp+"\n", timeout), cand))
    elapsed = time.time() - t0

    hits = [cand[i] for i, o in enumerate(outs) if o == exp]
    return elapsed, len(hits), (gold_idx[qi] in hits)

print("K=200, 8 workers\n")
times = []
for qi in np.where(has_ex)[0][:5]:
    r = rerank_timed(qi)
    if r:
        t, n, ok = r
        times.append(t)
        print(f"  {t:5.1f}s   {n} passed   gold found: {ok}")

print(f"\nmean: {np.mean(times):.1f}s per query")

K=200, 8 workers

   11.6s   1 passed   gold found: True
   12.3s   1 passed   gold found: True
   12.6s   1 passed   gold found: False
   12.0s   1 passed   gold found: False
   11.1s   1 passed   gold found: False

mean: 11.9s per query


### 10. Tuning the rerank cost

11 seconds per query is too slow. The hackathon judges speed explicitly.

Where the time goes: 200 subprocesses divided across 8 workers means 25 sequential rounds, each paying Python interpreter startup, plus the full timeout for anything that hangs.

Three knobs:

- **workers** — more parallelism, limited by available cores
- **timeout** — most wrong programs crash instantly, so a shorter cap mostly penalises genuine infinite loops
- **K** — fewer candidates, at the cost of recall

Target is under 2 seconds. If tuning alone can't reach it, the fix is architectural rather than a parameter change.

In [11]:
for w, tmo, K in [(8,2,200), (32,2,200), (32,1,200), (32,0.5,200), (32,1,100)]:
    ts = []
    for qi in np.where(has_ex)[0][:3]:
        r = rerank_timed(qi, K=K, workers=w, timeout=tmo)
        if r: ts.append(r[0])
    print(f"K={K:<4} workers={w:<3} timeout={tmo}s  ->  {np.mean(ts):5.1f}s")

K=200  workers=8   timeout=2s  ->   11.7s
K=200  workers=32  timeout=2s  ->   11.5s
K=200  workers=32  timeout=1s  ->    7.1s
K=200  workers=32  timeout=0.5s  ->    3.8s
K=100  workers=32  timeout=1s  ->    3.5s


### 11. Why more workers didn't help

Raising workers from 8 to 32 changed almost nothing, while halving the timeout nearly halved the runtime. That points at CPU cores as the bottleneck, not thread count.

Checking two things:

- how many cores are actually available
- what fraction of snippets run to the full timeout, rather than failing fast

In [12]:
import os
print("cores:", os.cpu_count())

# how many candidates actually hit the timeout?
qi = np.where(has_ex)[0][0]
m = pat.search(queries_t.text[qi])
inp = m.group(1).strip()
cand = np.argsort(-sim[qi])[:100]

slow = 0
durations = []
for j in cand:
    t0 = time.time()
    run_snippet(corpus_t.text[j], inp+"\n", timeout=2)
    d = time.time() - t0
    durations.append(d)
    if d > 1.8: slow += 1

print(f"hit timeout: {slow}/100")
print(f"median run:  {np.median(durations)*1000:.0f}ms")

cores: 2
hit timeout: 0/100
median run:  59ms


### 12. Right-sizing the worker count

Sequentially nothing times out and the median run is 57ms. Under 32 threads on 2 cores, lowering the timeout halved the runtime.

Both are only true if the threads are causing the timeouts. With 32 processes competing for 2 cores, each one is starved enough that wall-clock exceeds the limit and gets killed, even though it only needed 57ms of CPU.

Oversubscription is the problem. Testing worker counts near the core count instead.

In [13]:
for w, tmo, K in [(2,2,200), (4,2,200), (4,1,200), (2,1,200), (4,1,100), (4,1,50)]:
    ts, found = [], []
    for qi in np.where(has_ex)[0][:3]:
        r = rerank_timed(qi, K=K, workers=w, timeout=tmo)
        if r:
            ts.append(r[0]); found.append(r[2])
    print(f"K={K:<4} workers={w:<2} timeout={tmo}s  ->  {np.mean(ts):5.1f}s   gold found {sum(found)}/{len(found)}")

K=200  workers=2  timeout=2s  ->   13.2s   gold found 2/3
K=200  workers=4  timeout=2s  ->   12.6s   gold found 2/3
K=200  workers=4  timeout=1s  ->   11.2s   gold found 2/3
K=200  workers=2  timeout=1s  ->   12.1s   gold found 2/3
K=100  workers=4  timeout=1s  ->    5.4s   gold found 2/3
K=50   workers=4  timeout=1s  ->    3.1s   gold found 2/3


### 13. Early exit

Cost is linear in K at roughly 60ms per candidate, which matches the median run time. The bottleneck is Python interpreter startup, serialized across 2 cores. No parameter fixes that.

The fix is to run fewer programs. With a false positive rate of 0.3%, the first candidate producing correct output is almost certainly the answer, so there's no reason to keep going after a hit.

Running candidates in rank order and stopping at the first pass. If gold sits at rank 12, that costs 12 executions instead of 200.

In [14]:
def rerank_early_exit(qi, K=200, timeout=1):
    m = pat.search(queries_t.text[qi])
    if not m: return None
    inp, exp = m.group(1).strip(), m.group(2).strip()
    cand = np.argsort(-sim[qi])[:K]

    t0 = time.time()
    for n, j in enumerate(cand, 1):
        if run_snippet(corpus_t.text[j], inp+"\n", timeout) == exp:
            return time.time()-t0, n, (j == gold_idx[qi]), True
    return time.time()-t0, len(cand), False, False

print("K=200, early exit\n")
ts, ns = [], []
for qi in np.where(has_ex)[0][:8]:
    r = rerank_early_exit(qi)
    if r:
        t, n, ok, hit = r
        ts.append(t); ns.append(n)
        print(f"  {t:5.1f}s  after {n:3d} runs   gold: {ok}")

print(f"\nmean {np.mean(ts):.1f}s, mean {np.mean(ns):.0f} executions")

K=200, early exit

    2.6s  after  27 runs   gold: True
    2.5s  after  40 runs   gold: True
   16.2s  after 199 runs   gold: False
   14.1s  after 180 runs   gold: False
    6.0s  after  76 runs   gold: False
   14.7s  after 200 runs   gold: False
   16.5s  after 200 runs   gold: False
    0.3s  after   4 runs   gold: False

mean 9.1s, mean 116 executions


### 14. Where does gold sit inside the candidate list?

Early exit is fast when the gold is found and slow when it isn't, since a miss burns the full K.

Two things to measure before picking K:

- how deep the gold typically sits when it is present, which decides how much a smaller K actually costs
- how often a wrong snippet passes before the gold does, since top-ranked candidates are semantically similar and more likely to be near-duplicates than random ones were

In [15]:
in200 = np.where(has_ex & (rank <= 200))[0][:40]
pos = rank[in200]

print("gold rank among candidates (when in top-200):")
for k in [10, 25, 50, 100, 200]:
    print(f"  within top-{k:<4} {(pos<=k).mean()*100:5.1f}%")
print(f"\nmedian position: {int(np.median(pos))}")

# how often does something wrong pass first?
fp_first = 0
for qi in in200[:15]:
    r = rerank_early_exit(qi, K=200)
    if r and r[3] and not r[2]:
        fp_first += 1
print(f"\nwrong snippet passed before gold: {fp_first}/15")

gold rank among candidates (when in top-200):
  within top-10    35.0%
  within top-25    50.0%
  within top-50    75.0%
  within top-100   90.0%
  within top-200  100.0%

median position: 25

wrong snippet passed before gold: 7/15


### 15. Collecting all passers instead of stopping at the first

Early exit fails: among top-ranked candidates, a wrong snippet passes before the gold 47% of the time. Semantically similar problems share input format and output shape, so coincidental matches are common. Random snippets crash instead, which is why the earlier 0.3% figure was misleading.

New approach: run the full candidate set, collect every snippet that passes, and order those by embedding score. The gold lands near the top rather than exactly at rank 1, which is still a large gain over its original position.

Measuring how many pass per query, and where the gold ends up.

In [16]:
def rerank_all(qi, K=50, timeout=1):
    m = pat.search(queries_t.text[qi])
    if not m: return None
    inp, exp = m.group(1).strip(), m.group(2).strip()
    cand = np.argsort(-sim[qi])[:K]

    t0 = time.time()
    passers = [j for j in cand if run_snippet(corpus_t.text[j], inp+"\n", timeout) == exp]
    el = time.time()-t0

    # passers keep embedding order, then everyone else
    new_order = passers + [j for j in cand if j not in passers]
    pos = new_order.index(gold_idx[qi])+1 if gold_idx[qi] in new_order else 999
    return el, len(passers), pos, rank[qi]

print("K=50, all passers\n")
for qi in np.where(has_ex & (rank<=50))[0][:10]:
    r = rerank_all(qi)
    if r:
        el, np_, new, old = r
        print(f"  {el:4.1f}s  {np_:2d} passed   rank {old:3d} -> {new}")

K=50, all passers

   4.3s   1 passed   rank  27 -> 1
   3.3s   1 passed   rank  40 -> 1
   4.9s   0 passed   rank   4 -> 4
   4.8s   2 passed   rank  47 -> 2
   5.4s   1 passed   rank  21 -> 1
   3.5s   0 passed   rank   1 -> 1
   3.9s   3 passed   rank  27 -> 28
   3.1s   1 passed   rank   1 -> 1
   3.7s   1 passed   rank   3 -> 1
   3.0s   0 passed   rank  35 -> 35


### 16. Measuring the actual gain

Ten queries showed five large improvements, one regression, and four unchanged. Not enough to trust.

Running a larger sample and computing the NDCG@10 delta directly, including queries where the gold was never in the candidate set, since those are part of the real average.

In [17]:
N = 150
sample = np.where(has_ex)[0][:N]

old_r, new_r = [], []
t0 = time.time()
for qi in sample:
    r = rerank_all(qi, K=50)
    if r is None: continue
    _, _, new, old = r
    old_r.append(old); new_r.append(new if new != 999 else old)
total = time.time()-t0

old_r, new_r = np.array(old_r), np.array(new_r)
ndcg = lambda r: np.where(r<=10, 1/np.log2(r+1), 0).mean()*100

print(f"n = {len(old_r)}   ({total/len(old_r):.1f}s per query)\n")
print(f"  NDCG@10  before {ndcg(old_r):5.2f}   after {ndcg(new_r):5.2f}")
print(f"  MRR      before {(1/old_r).mean()*100:5.2f}   after {(1/new_r).mean()*100:5.2f}")
print(f"\n  improved {(new_r<old_r).sum()}, hurt {(new_r>old_r).sum()}, same {(new_r==old_r).sum()}")

n = 150   (3.6s per query)

  NDCG@10  before 12.78   after 30.74
  MRR      before 12.28   after 30.28

  improved 34, hurt 1, same 115
